In [14]:
import uuid
import json
import time

AGENT_CARD = {
    "id": "agent://text-cleaner",
    "methods": {
        "clean_text": {
            "description": "clean raw text",
            "input_spec": {
                "type": "object",
                "properties": {"raw_text": {"type": "string"}},
                "required": ["raw_text"],
            },
            "output_spec": {"type": "string"},
        }
    },
}


def create_task(method: str, params: dict) -> dict:
    return {
        "task_id": uuid.uuid4().hex,
        "method": method,
        "input": params,
        "status": "pending",
        "created_at": time.time(),
    }


def execute_task(task: dict) -> dict:
    if task["method"] != "clean_text":
        return {
            "task_id": task["task_id"],
            "status": "error",
            "error": f"unsupported method {task["method"]}",
        }

    raw_text = task["input"]["raw_text"]
    cleaned = "".join(c for c in raw_text if c.isalnum() or c.isspace())
    stop_words = ["a", "the", "this", "and", "is", "to", "in"]
    cleaned_words = [w for w in cleaned.split() if w.lower() not in stop_words]
    result = "".join(cleaned_words)

    return {
        "task_id": task["task_id"],
        "status": "completed",
        "output": result
    }

def json_rpc_request(method:str, params:dict, id:int=1) -> dict:
    return {
        "jsonrpc":"2.0",
        "method":method,
        "params":params,
        "id":id
    }

In [ ]:
# test

task = create_task("clean_text", {"raw_text": "this is a testing task"})
result = execute_task(task)
rpc = json_rpc_request("clean_text", {"text": "try this input in json prc format"})


print(f"agent card:{AGENT_CARD}\ntask:{task}\nresult:{result}\rpc:{rpc}")

agent card:{'id': 'agent://text-cleaner', 'methods': {'clean_text': {'description': 'clean raw text', 'input_spec': {'type': 'object', 'properties': {'raw_text': {'type': 'string'}}, 'required': ['raw_text']}, 'output_spec': {'type': 'string'}}}}
task:{'task_id': '2fe0ac875c5046efb265ac6896f82c8c', 'method': 'clean_text', 'input': {'raw_text': 'this is a testing task'}, 'status': 'pending', 'created_at': 1791186543.9797184}
pc:{'jsonrpc': '2.0', 'method': 'clean_text', 'params': {'text': 'try this input in json prc format'}, 'id': 1}


: 